In [1]:
# CONFIGURATION
RAW_PATH = "data/nsrdb_raw.csv"
OUTPUT_PATH = "data/nsrdb_preprocessed.csv"
INTERP_LIMIT = 3
WINDOW_SIZE = 24
RANDOM_SEED = 42

import numpy as np
import pandas as pd
import joblib
import os
from sklearn.preprocessing import MinMaxScaler

np.random.seed(RANDOM_SEED)
os.makedirs("data", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

In [2]:
# LOAD
df = pd.read_csv(RAW_PATH, index_col="datetime", parse_dates=True)
print(f"Loaded : {df.shape} | {df.index.min()} > {df.index.max()}")
print(f"\nColumn names as loaded:\n{df.columns.tolist()}")

Loaded : (87672, 10) | 2015-01-01 00:30:00 > 2024-12-31 23:30:00

Column names as loaded:
['GHI', 'DNI', 'DHI', 'Temperature', 'Relative Humidity', 'Dew Point', 'Wind Speed', 'Wind Direction', 'Surface Albedo', 'Solar Zenith Angle']


In [3]:
# STANDARDIZE COLUMN NAMES
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
print(df.columns.tolist())

['ghi', 'dni', 'dhi', 'temperature', 'relative_humidity', 'dew_point', 'wind_speed', 'wind_direction', 'surface_albedo', 'solar_zenith_angle']


In [4]:
# DATA QUALITY CHECK
print("Missing Values")
missing = df.isnull().mean() * 100
print(missing.round(4))

print("\nOut of range physical checks")
checks = {
    "ghi < 0": (df["ghi"] < 0).sum(),
    "dni < 0": (df["dni"] < 0).sum(),
    "dhi < 0": (df["dhi"] < 0).sum(),
    "relative_humidity < 0": (df["relative_humidity"] < 0).sum(),
    "solar_zenith_angle < 0": (df["solar_zenith_angle"] < 0).sum(),
}

for check, count in checks.items():
    print(f"{check}: {count} rows")

Missing Values
ghi                   0.0
dni                   0.0
dhi                   0.0
temperature           0.0
relative_humidity     0.0
dew_point             0.0
wind_speed            0.0
wind_direction        0.0
surface_albedo        0.0
solar_zenith_angle    0.0
dtype: float64

Out of range physical checks
ghi < 0: 0 rows
dni < 0: 0 rows
dhi < 0: 0 rows
relative_humidity < 0: 0 rows
solar_zenith_angle < 0: 0 rows


In [5]:
# HANDLE MISSING VALUES
# Flag columns with >1% missing before interpolating
high_missing = missing[missing > 1.0]
if not high_missing.empty:
    print("WARNING - columns with >1% missing, review before proceeding")
    print(high_missing)
else:
    print("All columns under 1% missing threhold")

# Linear interpolation for gaps of INTERP_LIMIT hours or fewer
before = df.isnull().sum().sum()
df = df.interpolate(method="time", limit=INTERP_LIMIT)
after = df.isnull().sum().sum()
print(f"\nImputed {before - after} values via linear interpolation")
print(f"Remaining nulls after interpolation: {after}")

All columns under 1% missing threhold

Imputed 0 values via linear interpolation
Remaining nulls after interpolation: 0


In [6]:
# FEATURE ENGINEERING
# Cyclical time encoding preserve periodicty for the LSTM.
# Categorical style time variables (hours, day of year, month) are encoded as sine/cosine pairs so the model sees continuity across period boundaries.

df["hour_sin"] = np.sin(2 * np.pi * df.index.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df.index.hour / 24)
df["doy_sin"] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)
df["month_sin"] = np.sin(2 * np.pi * df.index.month / 12)
df["month_cos"] = np.cos(2 * np.pi * df.index.month / 12)

# Nighttime flag - GHI is 0 at night, zenith > 90 means sun below horizon
df["is_daytime"] = ((df["ghi"] > 0) & (df["solar_zenith_angle"] < 90)).astype(int)

print(f"Feature after engineering: {df.columns.tolist()}")
print(f"Shape: {df.shape}")

Feature after engineering: ['ghi', 'dni', 'dhi', 'temperature', 'relative_humidity', 'dew_point', 'wind_speed', 'wind_direction', 'surface_albedo', 'solar_zenith_angle', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'month_sin', 'month_cos', 'is_daytime']
Shape: (87672, 17)


In [7]:
# TRAIN/VAL/TEST SPLIT (70/15/15, time ordered, no shuffle)
n = len(df)
train_end = int(n*.7)
val_end = int(n*.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(f"Train: {train_df.index.min().date()} > {train_df.index.max().date()} ({len(train_df):,} rows)")
print(f"Val: {val_df.index.min().date()} > {val_df.index.max().date()} ({len(val_df):,} rows)")
print(f"Test: {test_df.index.min().date()} > {test_df.index.max().date()} ({len(test_df):,} rows)")

Train: 2015-01-01 > 2022-01-01 (61,370 rows)
Val: 2022-01-01 > 2023-07-03 (13,151 rows)
Test: 2023-07-03 > 2024-12-31 (13,151 rows)


In [8]:
# HOURLY MINMAX SCALING (fit on training only - no leakage)
scaler = MinMaxScaler()
feature_cols = df.columns.tolist()

train_scaled = scaler.fit_transform(train_df[feature_cols])
val_scaled = scaler.transform(val_df[feature_cols])
test_scaled = scaler.transform(test_df[feature_cols])

# Rebuild as DataFrames with datetime index preserved
train_scaled = pd.DataFrame(train_scaled, index=train_df.index, columns=feature_cols)
val_scaled = pd.DataFrame(val_scaled, index=val_df.index, columns=feature_cols)
test_scaled = pd.DataFrame(test_scaled, index=test_df.index, columns=feature_cols)

print("Hourly scaling complete - fit on training only (no leakage)")
print(f"\nTrain scaled range check (should be 0-1):")
print(train_scaled.describe().loc[["min", "max"]])

Hourly scaling complete - fit on training only (no leakage)

Train scaled range check (should be 0-1):
     ghi  dni  dhi  temperature  relative_humidity  dew_point  wind_speed  \
min  0.0  0.0  0.0          0.0                0.0        0.0         0.0   
max  1.0  1.0  1.0          1.0                1.0        1.0         1.0   

     wind_direction  surface_albedo  solar_zenith_angle  hour_sin  hour_cos  \
min             0.0             0.0                 0.0       0.0       0.0   
max             1.0             1.0                 1.0       1.0       1.0   

     doy_sin  doy_cos  month_sin  month_cos  is_daytime  
min      0.0      0.0        0.0        0.0         0.0  
max      1.0      1.0        1.0        1.0         1.0  


In [9]:
# DAILY RESAMPLE (for medium/long horizon models)
# GHI/DNI/DHI summed to daily energy (Wh/m^2)
DAILY_OUTPUT_PATH = "data/nsrdb_daily.csv"

df_daily = df.resample("D").agg({
    "ghi": "sum",
    "dni": "sum",
    "dhi": "sum",
    "temperature": "mean",
    "relative_humidity": "mean",
    "dew_point": "mean",
    "wind_speed": "mean",
    "wind_direction": "mean",
    "surface_albedo": "mean",
    "solar_zenith_angle": "mean",
    "doy_sin": "first",
    "doy_cos": "first",
    "month_sin": "first",
    "month_cos": "first",
    "is_daytime": "sum",   
})

# Rename is_daytime to daylight_hours for clarity at daily resolution
df_daily.rename(columns={"is_daytime": "daylight_hours", "air_temperature": "tempreature"}, inplace=True)

# Drop hour cyclicals - not meaningful at daily resolution
df_daily.drop(columns=["hour_sin", "hour_cos"], inplace=True, errors="ignore")

# Time ordered 70/15/15 split (same Boundaries as hourly)
n_d = len(df_daily)
daily_train = df_daily.iloc[:int(n_d*.7)]
daily_val = df_daily.iloc[int(n_d*.7): int(n_d*.85)]
daily_test = df_daily.iloc[int(n_d*.85):]

print(f"Daily dataset: {df_daily.shape}")
print(f"Date range: {df_daily.index.min().date()} > {df_daily.index.max().date()}")
print(f"Columns: {df_daily.columns.tolist()}")
print(f"Daily GHI range (Wh/m^2): {df_daily['ghi'].min():.0f} > {df_daily['ghi'].max():.0f}")
print(f"\nSplit sizes:")
print(f"Train: {daily_train.index.min().date()} > {daily_train.index.max().date()} ({len(daily_train)} days)")
print(f"Val: {daily_val.index.min().date()} > {daily_val.index.max().date()} ({len(daily_val)} days)")
print(f"Test: {daily_test.index.min().date()} > {daily_test.index.max().date()} ({len(daily_test)} days)")


Daily dataset: (3653, 15)
Date range: 2015-01-01 > 2024-12-31
Columns: ['ghi', 'dni', 'dhi', 'temperature', 'relative_humidity', 'dew_point', 'wind_speed', 'wind_direction', 'surface_albedo', 'solar_zenith_angle', 'doy_sin', 'doy_cos', 'month_sin', 'month_cos', 'daylight_hours']
Daily GHI range (Wh/m^2): 143 > 9023

Split sizes:
Train: 2015-01-01 > 2021-12-31 (2557 days)
Val: 2022-01-01 > 2023-07-02 (548 days)
Test: 2023-07-03 > 2024-12-31 (548 days)


In [10]:
# DAILY MINMAX SCALING (fit on daily training only - no leakage)
daily_feature_cols = df_daily.columns.tolist()

daily_scaler = MinMaxScaler()
daily_train_scaled = daily_scaler.fit_transform(daily_train[daily_feature_cols])
daily_val_scaled = daily_scaler.transform(daily_val[daily_feature_cols])
daily_test_scaled = daily_scaler.transform(daily_test[daily_feature_cols])

# Rebuild as DataFrames preserving datetime index
daily_train_scaled = pd.DataFrame(daily_train_scaled, index=daily_train.index, columns=daily_feature_cols)
daily_val_scaled = pd.DataFrame(daily_val_scaled, index=daily_val.index, columns=daily_feature_cols)
daily_test_scaled = pd.DataFrame(daily_test_scaled, index=daily_test.index, columns=daily_feature_cols)

print("Daily scaling complete - fit on training only (no leakage)")
print(f"Train range check: min={daily_train_scaled.min().min():.2f}, max={daily_train_scaled.max().max():.2f}")

Daily scaling complete - fit on training only (no leakage)
Train range check: min=0.00, max=1.00


In [11]:
# SAVE ALL OUTPUTS
# Full preprocessed unscaled dataset (used by Prophet & XGBoost directly)
df.to_csv(OUTPUT_PATH)
print(f"\nPreprocessed dataset saved to {OUTPUT_PATH}")

# Hourly scaled splits for LSTM
train_scaled.to_csv("data/training_scaled.csv")
val_scaled.to_csv("data/val_scaled.csv")
test_scaled.to_csv("data/test_scaled.csv")

# Daily unscaled splits
daily_train.to_csv("data/nsrdb_daily_train.csv")
daily_val.to_csv("data/nsrdb_daily_val.csv")
daily_test.to_csv("data/nsrdb_daily_test.csv")
df_daily.to_csv(DAILY_OUTPUT_PATH)

# Daily scaled splits for XGBoost medium horizon models
daily_train_scaled.to_csv("data/daily_train_scaled.csv")
daily_val_scaled.to_csv("data/daily_val_scaled.csv")
daily_test_scaled.to_csv("data/daily_test_scaled.csv")

# Fitted scalers - needed to inverse transform predictions back to W/m^2
joblib.dump(scaler, "artifacts/minmax_scaler.pkl")
joblib.dump(daily_scaler, "artifacts/daily_minmax_scaler.pkl")
print("Hourly scaler saved to artifacts/minmax_scaler.pkl")
print("Daily scaler saved to artifacts/daily_minmax_scaler.pkl")


Preprocessed dataset saved to data/nsrdb_preprocessed.csv


Hourly scaler saved to artifacts/minmax_scaler.pkl
Daily scaler saved to artifacts/daily_minmax_scaler.pkl


In [12]:
# SUMMARY
print(f"Preprocessing Summary")
print(f"Total rows: {len(df):,}")
print(f"Features: {len(feature_cols)} ({', '.join(feature_cols)})")
print(f"Train/Val/Test: {len(train_df):,}/{len(val_df):,}/{len(test_df):,}")
print(f"GHI range (W/m^2): {df['ghi'].min():.1f} > {df['ghi'].max():.1f}")
print(f"Peak GHI: {df['ghi'].max():.1f} W/m^2")
print(f"10% RMSE target: {df['ghi'].max() * .1:.1f} W/m^2")
print(f"Daily rows: {len(df_daily):,}")
print(f"Daily features: {len(daily_feature_cols)}")

Preprocessing Summary
Total rows: 87,672
Features: 17 (ghi, dni, dhi, temperature, relative_humidity, dew_point, wind_speed, wind_direction, surface_albedo, solar_zenith_angle, hour_sin, hour_cos, doy_sin, doy_cos, month_sin, month_cos, is_daytime)
Train/Val/Test: 61,370/13,151/13,151
GHI range (W/m^2): 0.0 > 990.0
Peak GHI: 990.0 W/m^2
10% RMSE target: 99.0 W/m^2
Daily rows: 3,653
Daily features: 15
